# 6. Bonus: is your variational inference actually right?

Two experiments on Bayesian linear regression, the one model here whose posterior
can be written down exactly. Because the answer is known, every claim in this
notebook is checkable rather than plausible.

**(a)** Mean-field against full covariance, as the features become correlated.
This turns the standard warning about mean-field variational inference into a
number, and the number has a closed form to compare against.

**(b)** Two gradient estimators for the same objective. Both are unbiased. One has
a few hundred times the variance of the other.

You write `fit_vi_gaussian` and `gradient_estimates`. About 4 hours. Runs in well
under a minute.

## 6.1 The exact answer

With `y = X w + noise`, `noise ~ N(0, s_n^2)` and prior `w ~ N(0, s_p^2 I)`, the
posterior is Gaussian:

    precision = X^T X / s_n^2  +  I / s_p^2
    Sigma     = inverse(precision)
    mean      = Sigma X^T y / s_n^2

Complete the square in the exponent of the product of prior and likelihood if you
want to see where that comes from.

Part (a) optimises the ELBO in closed form, with no sampling at all, because the
question there is about the variational *family*, not about estimators. Part (b) is
where estimators are the subject.

In [ ]:
import sys

sys.path.insert(0, "..")

import math

import matplotlib.pyplot as plt
import numpy as np
import torch

from bdl.store import run_dir

%matplotlib inline


def make_correlated_data(n=200, d=4, rho=0.0, noise_std=0.3, seed=0):
    """Linear-regression data whose features have pairwise correlation rho.

    Correlated features are what make the posterior correlated, which is what
    mean-field cannot represent. rho = 0 is the easy case.
    """
    g = torch.Generator().manual_seed(seed)
    cov = torch.full((d, d), rho) + (1.0 - rho) * torch.eye(d)
    chol = torch.linalg.cholesky(cov)
    x = torch.randn(n, d, generator=g) @ chol.T
    w_true = torch.randn(d, generator=g)
    y = x @ w_true + noise_std * torch.randn(n, generator=g)
    return x, y.reshape(-1, 1), w_true


def analytic_posterior(x, y, noise_std, prior_std):
    """The exact posterior mean and covariance."""
    d = x.shape[1]
    precision = x.T @ x / noise_std**2 + torch.eye(d) / prior_std**2
    cov = torch.linalg.inv(precision)
    mean = cov @ x.T @ y.reshape(-1) / noise_std**2
    return mean, cov


def exact_elbo_terms(q_mean, q_cov, x, y, noise_std, prior_std):
    """The negative ELBO for a Gaussian q, in closed form. No sampling."""
    n, d = x.shape
    resid = y.reshape(-1) - x @ q_mean
    # E_q[||y - Xw||^2] = ||y - X mu||^2 + trace(X Sigma X^T)
    quad = (resid**2).sum() + torch.einsum("ij,jk,ik->", x, q_cov, x)
    nll = 0.5 * quad / noise_std**2 + n * math.log(noise_std * math.sqrt(2 * math.pi))
    kl = 0.5 * (
        (torch.trace(q_cov) + q_mean @ q_mean) / prior_std**2
        - d
        + 2 * d * math.log(prior_std)
        - torch.logdet(q_cov)
    )
    return nll + kl

## 6.2 Part (a): the two families

Fit a Gaussian `q` to the posterior by minimising the exact negative ELBO, in two
families:

* **mean-field**: learn `mean` of shape `[d]` and `log_sigma` of shape `[d]`; the
  covariance is `diag(exp(2 * log_sigma))`;
* **full covariance**: learn `mean` and an unconstrained lower-triangular `L` of
  shape `[d, d]`, and take `Sigma = L L^T` with `exp` applied to the diagonal of
  `L` so it stays positive. That is a Cholesky parameterisation, the standard way
  to keep a learned covariance positive-definite.

What to expect. Both recover the posterior *mean* almost exactly. The
full-covariance fit also recovers the covariance. The mean-field fit returns
variances matching `1 / diag(precision)` rather than `diag(inverse(precision))`,
which is smaller whenever the features are correlated.

**Run the control first.** The full-covariance fit must reproduce the exact answer
to within about a percent before anything the mean-field fit tells you can be
trusted. At 2000 optimisation steps *both* families come out about 13% too wide,
which looks exactly like a mean-field pathology and is nothing but
under-convergence. The default here is 5000.

In [ ]:
def fit_vi_gaussian(x, y, noise_std, prior_std, full_covariance, steps=5000, lr=5e-2):
    """Fit a Gaussian q by minimising the exact ELBO. Returns (mean, cov), detached."""
    # ---- TODO ------------------------------------------------------------
    # parameterise q:
    #   mean-field       mean [d], log_sigma [d]     -> diag(exp(2 log_sigma))
    #   full covariance  mean [d], raw_l [d, d]      -> L L^T, with exp on diag(L)
    # then minimise exact_elbo_terms(mean, cov, x, y, noise_std, prior_std) with Adam
    raise NotImplementedError
    # ----------------------------------------------------------------------

In [ ]:
# ---- check your work: the control ------------------------------------------
x, y, _ = make_correlated_data(rho=0.8, seed=0)
noise_std, prior_std = 0.3, 1.0
exact_mean, exact_cov = analytic_posterior(x, y, noise_std, prior_std)
exact_std = torch.sqrt(torch.diag(exact_cov))

fc_mean, fc_cov = fit_vi_gaussian(x, y, noise_std, prior_std, full_covariance=True)
fc_std = torch.sqrt(torch.diag(fc_cov))

mean_err = float((fc_mean - exact_mean).abs().max())
std_ratio = float((fc_std / exact_std).mean())
off_diag = float((fc_cov - exact_cov).abs().max() / exact_cov.abs().max())

print(f"full covariance: max mean error {mean_err:.5f}")
print(f"                 std / exact    {std_ratio:.4f}   (must be ~1.000)")
print(f"                 max covariance error, relative {off_diag:.4f}")
assert mean_err < 0.01, "full-covariance VI did not find the posterior mean"
assert abs(std_ratio - 1.0) < 0.01, (
    f"full-covariance VI is {std_ratio:.3f} of the exact width: an optimisation "
    "problem, not a statistics one. Increase steps before continuing."
)
print("\nOK   the control passes, so the mean-field measurement below is meaningful")

## 6.3 The sweep, and the closed form

Now sweep the feature correlation. Alongside the two fits, compute what theory
says mean-field should return. Minimising `KL(q || p)` over a diagonal `q` for a
Gaussian target gives, coordinate by coordinate,

    mean_i    = exact mean_i          (exactly right)
    sigma_i^2 = 1 / precision_ii      (not Sigma_ii)

and by the Schur complement `Sigma_ii * precision_ii = 1 / (1 - R_i^2)`, where
`R_i` is the multiple correlation of coordinate `i` with the others under the
posterior. So the ratio of widths is

    mean-field std / exact std  =  sqrt(1 - R_i^2)   <=  1

with equality only when the posterior is uncorrelated. The measured curve should
sit on top of that line.

In [ ]:
noise_std, prior_std = 0.3, 1.0
rhos = (0.0, 0.3, 0.6, 0.8, 0.9, 0.95)
rows = []

for rho in rhos:
    x, y, _ = make_correlated_data(rho=rho, seed=0)
    exact_mean, exact_cov = analytic_posterior(x, y, noise_std, prior_std)
    exact_std = torch.sqrt(torch.diag(exact_cov))
    precision = torch.linalg.inv(exact_cov)

    mf_mean, mf_cov = fit_vi_gaussian(x, y, noise_std, prior_std, full_covariance=False)
    fc_mean, fc_cov = fit_vi_gaussian(x, y, noise_std, prior_std, full_covariance=True)

    r2 = 1.0 - 1.0 / (torch.diag(exact_cov) * torch.diag(precision))
    rows.append(
        {
            "rho": rho,
            "mean_field": float((torch.sqrt(torch.diag(mf_cov)) / exact_std).mean()),
            "full_cov": float((torch.sqrt(torch.diag(fc_cov)) / exact_std).mean()),
            "closed_form": float(torch.sqrt(1.0 - r2).mean()),
            "r2": float(r2.mean()),
            "mf_mean_err": float((mf_mean - exact_mean).abs().max()),
        }
    )
    r = rows[-1]
    print(
        f"rho={rho:4.2f}   mean-field {r['mean_field']:.3f}   "
        f"closed form {r['closed_form']:.3f}   full cov {r['full_cov']:.3f}   "
        f"R^2 {r['r2']:.3f}   mean error {r['mf_mean_err']:.4f}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 4.2))
r = [row["rho"] for row in rows]
ax.plot(r, [row["mean_field"] for row in rows], "o-", label="mean-field VI")
ax.plot(r, [row["full_cov"] for row in rows], "s-", label="full-covariance VI")
ax.plot(r, [row["closed_form"] for row in rows], "k:", label=r"closed form, $\sqrt{1-R^2}$")
ax.axhline(1.0, color="0.5", lw=1)
ax.set_xlabel("feature correlation")
ax.set_ylabel("posterior std / exact posterior std")
ax.set_title("Mean-field VI underestimates the posterior width\n(below 1 = overconfident)")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(run_dir("bonus_vi") / "mean_field_shrinkage.png", dpi=150, bbox_inches="tight")

At a correlation of 0.95 the credible intervals are about three times too narrow,
and the measured curve matches the closed form to three decimals. The mechanism is
the one from notebook 04: the objective minimises `KL(q || p)`, which is finite
only where `q` puts mass where `p` does, so a `q` that cannot tilt prefers to sit
inside the correlated ridge rather than straddle it.

For the report: state the control result first, then the measurement, then the
closed form. A shrinkage curve without the control is not evidence of anything.

## 6.4 Part (b): two ways to differentiate an expectation

The ELBO is `E_q[f(w)]` with `f(w) = log p(D, w) - log q(w)`, and the parameters we
differentiate with respect to sit inside `q`. Two estimators.

**Reparameterization.** Write `w = mean + sigma * eps` with `eps ~ N(0, I)`. The
measure no longer depends on the parameters, so

    grad E_q[f] = E_eps[ grad f(mean + sigma * eps) ]

and one backward pass through `f` gives the gradient. This is what notebook 04
used.

**Score function**, also called the log-derivative trick or REINFORCE. Using
`grad q = q * grad log q`,

    grad E_q[f] = E_q[ f(w) * grad log q(w) ]

which needs no derivative of `f` at all: `f` may be a black box, discrete, or
non-differentiable.

Both are unbiased estimators of the same quantity. Implement both and measure
their variance against the number of Monte Carlo samples.

For a Gaussian `q` with mean `mean`, `grad_mean log q = eps / sigma`, so the
score-function estimate is proportional to the *magnitude* of `f`, and `f` contains
`log p(D | w)`, which is of order `N`. The reparameterized estimate depends on the
variation of `grad f` instead. Predict which one has the larger variance before you
run it.

In [ ]:
def elbo_sample_terms(w, x, y, noise_std, prior_std):
    """log p(D, w) for a batch of sampled weight vectors w of shape [S, d]."""
    resid = y.reshape(1, -1) - w @ x.T
    log_lik = -0.5 * (resid**2).sum(-1) / noise_std**2
    log_prior = -0.5 * (w**2).sum(-1) / prior_std**2
    return log_lik + log_prior

In [ ]:
def gradient_estimates(estimator, mean, log_sigma, x, y, noise_std, prior_std, n_mc, generator):
    """One Monte Carlo estimate of the ELBO gradient with respect to `mean`. Returns [d].

    Both estimators start by drawing eps ~ N(0, I) of shape [n_mc, d] and forming
    w = mean + exp(log_sigma) * eps.
    """
    # ---- TODO ------------------------------------------------------------
    # "reparam": f(w) = elbo_sample_terms(w, ...) - log q(w) is differentiable in
    #            w, and w is differentiable in mean, so backpropagate -f.mean()
    #            and read off mean.grad.
    #
    # "score":   pretend you cannot differentiate through the sampling. Detach w,
    #            evaluate f as a plain number, compute log q(w) as a function of
    #            mean, and backpropagate -(f.detach() * log_q).mean().
    raise NotImplementedError
    # ----------------------------------------------------------------------

In [ ]:
# ---- check your work: both estimate the same gradient ----------------------
# A variance comparison between estimators of *different* quantities means
# nothing, so check the means agree before comparing spreads.
noise_std_b, prior_std_b = 0.3, 4.0
xb, yb, _ = make_correlated_data(n=100, d=4, rho=0.0, noise_std=noise_std_b, seed=1)
mean_b = torch.full((xb.shape[1],), 0.3)  # a fixed, non-optimal point
log_sigma_b = torch.full((xb.shape[1],), -1.0)

means = {}
for estimator in ("reparam", "score"):
    g = torch.Generator().manual_seed(7)
    grads = torch.stack(
        [
            gradient_estimates(
                estimator, mean_b, log_sigma_b, xb, yb, noise_std_b, prior_std_b, 200, g
            )
            for _ in range(200)
        ]
    )
    means[estimator] = grads.mean(0)
    print(f"{estimator:8} mean gradient {[f'{v:+.1f}' for v in means[estimator].tolist()]}")

rel = float((means["reparam"] - means["score"]).norm() / means["reparam"].norm())
assert rel < 0.05, f"the two estimators disagree by {rel:.1%}: one of them is biased"
print(f"\nOK   the two means agree to {rel:.2%}, so both are estimating the same gradient")

In [ ]:
n_mc_values = (1, 2, 5, 10, 20, 50, 100)
n_repeats = 200
rows_b = []

for n_mc in n_mc_values:
    row = {"n_mc": n_mc}
    for estimator in ("reparam", "score"):
        g = torch.Generator().manual_seed(7)
        grads = torch.stack(
            [
                gradient_estimates(
                    estimator, mean_b, log_sigma_b, xb, yb, noise_std_b, prior_std_b, n_mc, g
                )
                for _ in range(n_repeats)
            ]
        )
        row[f"{estimator}_var"] = float(grads.var(dim=0, unbiased=True).sum())
    rows_b.append(row)
    print(
        f"n_mc={n_mc:4d}   var(reparam)={row['reparam_var']:.3e}   "
        f"var(score)={row['score_var']:.3e}   ratio={row['score_var'] / row['reparam_var']:.0f}x"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 4.2))
ns = [r["n_mc"] for r in rows_b]
ax.loglog(ns, [r["reparam_var"] for r in rows_b], "o-", label="reparameterization")
ax.loglog(ns, [r["score_var"] for r in rows_b], "s-", label="score function")
ax.set_xlabel("Monte Carlo samples per gradient estimate")
ax.set_ylabel("variance of the ELBO gradient")
ax.set_title("Why variational inference is written with the reparameterization trick")
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig(run_dir("bonus_vi") / "gradient_variance.png", dpi=150, bbox_inches="tight")

ratio = np.mean([r["score_var"] / r["reparam_var"] for r in rows_b])
print(f"mean variance ratio across budgets: {ratio:.0f}x")
print(f"to match one reparameterized sample you would need about {ratio:.0f} score-function samples")

## 6.5 What to report

* the control: full-covariance VI recovering the exact posterior, stated before any
  mean-field claim;
* the shrinkage curve against the closed form, and what the ratio is at the
  strongest correlation;
* the variance ratio, with evidence that both estimators have the same mean, and
  what the ratio implies for the number of samples you would need;
* both variances fall as `1 / n_mc`. Say what that does and does not fix: it
  reduces the variance of an estimator that starts a few hundred times worse, and
  it grows with the dataset size and the dimension, which is why nobody trains a
  neural network this way.

One extension worth a paragraph if you have time. The score-function estimator
admits a control variate: since `E[grad log q] = 0`, subtracting any constant `b`
from `f` leaves the estimator unbiased, and taking `b` near `E[f]` replaces
`E[f^2]` by `Var[f]` in the variance. Measure how much that recovers.